# Docling layout exploration

Experiment with how [Docling](https://github.com/docling-project/docling) labels document structure on clinical protocol PDFs before building the SoA locator/extractor.

**What to look for when comparing to the source PDF:**
- Are section headings (especially SoA-related titles) detected?
- Are large tables split across pages or merged incorrectly?
- Do hierarchical row/column headers survive in table grids?
- Are footnotes captured and linked, including continuations past page breaks?

## 1. Configuration

In [2]:
from __future__ import annotations

import json
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions
from docling.datamodel.pipeline_options import AcceleratorDevice
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import TableItem, TextItem
from docling_core.types.doc.labels import DocItemLabel

# Paths — notebook lives in intake-soa/notebooks/
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROTOCOLS_DIR = (PROJECT_DIR / ".." / "takehome-1b").resolve()
OUTPUT_DIR = (PROJECT_DIR / "outputs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Swap to protocol5.pdf, protocol9.pdf, protocol12.pdf, or protocol15.pdf
PROTOCOL = PROTOCOLS_DIR / "protocol1.pdf"

# Pipeline toggles
DO_TABLE_STRUCTURE = True
DO_OCR = False  # enable for scanned pages

SOA_HEADING_KEYWORDS = (
    "schedule of activities",
    "schedule of assessments",
    "study flow",
    "table of events",
    "time and events",
)

print(f"Protocols dir: {PROTOCOLS_DIR}")
print(f"Exists: {PROTOCOLS_DIR.is_dir()}")
print(f"Selected PDF: {PROTOCOL}")
print(f"Exists: {PROTOCOL.is_file()}")
print(f"Output dir: {OUTPUT_DIR}")

Protocols dir: /Users/rushi/Desktop/Rushi/Assessment/takehome-1b
Exists: True
Selected PDF: /Users/rushi/Desktop/Rushi/Assessment/takehome-1b/protocol1.pdf
Exists: True
Output dir: /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs


## 2. Convert one PDF

In [3]:
if not PROTOCOL.is_file():
    raise FileNotFoundError(
        f"Protocol not found: {PROTOCOL}\n"
        "Unzip takehome-1b.zip into the repo root so ../takehome-1b/ exists."
    )

pipeline_options = PdfPipelineOptions(
    do_table_structure=DO_TABLE_STRUCTURE,
    do_ocr=DO_OCR,
    # Avoid macOS kernel segfaults during model inference (Cursor/Jupyter crash).
    accelerator_options=AcceleratorOptions(num_threads=1, device=AcceleratorDevice.CPU),
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

t0 = time.perf_counter()
result = converter.convert(str(PROTOCOL))
elapsed = time.perf_counter() - t0

doc = result.document
status = getattr(result, "status", "unknown")
page_count = len(getattr(doc, "pages", {}) or {})

print(f"Conversion status: {status}")
print(f"Elapsed: {elapsed:.1f}s")
print(f"Document name: {doc.name}")
print(f"Pages: {page_count}")
print(f"Text items: {len(doc.texts)}")
print(f"Tables: {len(doc.tables)}")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-09-02 22:25:58,153 MatchingPostProcessor WARNING  Orphan pdf_cell 124 recovered to row=21 by nearest-row fallback (col=0, y=856.6, dist=25.9)
2026-09-02 22:26:02,036 MatchingPostProcessor WARNING  Orphan pdf_cell 148 recovered to row=23 by nearest-row fallback (col=0, y=1044.4, dist=41.0)
2026-09-02 22:26:02,036 MatchingPostProcessor WARNING  Orphan pdf_cell 149 recovered to row=23 by nearest-row fallback (col=0, y=1044.4, dist=41.0)
2026-09-02 22:26:02,036 MatchingPostProcessor WARNING  Orphan pdf_cell 150 recovered to row=24 by nearest-row fallback (col=0, y=1070.4, dist=27.6)
2026-09-02 22:26:02,036 MatchingPostProcessor WARNING  Orphan pdf_cell 151 recovered to row=24 by nearest-row fallback (col=0, y=1070.4, dist=27.6)
Conversion status: ConversionStatus.PARTIAL_SUCCESS
Elapsed: 36.2s
Document name: protocol1
Pages: 97
Text items: 1278
Tables: 21


## 3. Label inventory (layout classification)

Tally how Docling classifies each element. This is the main view of layout labeling quality.

In [4]:
def item_label(item) -> str:
    label = getattr(item, "label", None)
    if label is None:
        return type(item).__name__
    return label.value if hasattr(label, "value") else str(label)


def item_page(item) -> int | None:
    prov = getattr(item, "prov", None) or []
    if not prov:
        return None
    return getattr(prov[0], "page_no", None)


label_counts: Counter[str] = Counter()
type_counts: Counter[str] = Counter()

for item, _level in doc.iterate_items():
    label_counts[item_label(item)] += 1
    type_counts[type(item).__name__] += 1

label_df = (
    pd.DataFrame(
        [{"label": k, "count": v} for k, v in label_counts.most_common()]
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Labels (DocItemLabel / inferred):")
display(label_df)

print("\nPython types:")
display(pd.DataFrame([{"type": k, "count": v} for k, v in type_counts.most_common()]))

Labels (DocItemLabel / inferred):


,label,count
0,text,478
1,list_item,266
2,section_header,194
3,table,19
4,code,16
5,checkbox_unselected,7
6,picture,5
7,footnote,4
8,document_index,2
9,caption,1



Python types:


,type,count
0,TextItem,490
1,ListItem,266
2,SectionHeaderItem,194
3,TableItem,21
4,CodeItem,16
5,PictureItem,5


## 4. Reading order + hierarchy

First items in document order. Section headers may hint where the SoA lives.

In [5]:
MAX_ITEMS = 80
soa_heading_hits: list[tuple[int, int, str]] = []

for i, (item, level) in enumerate(doc.iterate_items()):
    if i >= MAX_ITEMS:
        break

    label = item_label(item)
    page = item_page(item)
    text = ""
    if isinstance(item, TextItem):
        text = (item.text or "").strip().replace("\n", " ")
        lower = text.lower()
        if any(kw in lower for kw in SOA_HEADING_KEYWORDS):
            soa_heading_hits.append((page or -1, level, text))
    elif isinstance(item, TableItem):
        text = "[TABLE]"

    snippet = text[:120] + ("…" if len(text) > 120 else "")
    print(f"{i:03d}  L{level}  p{page}  {label:20s}  {snippet}")

print("\n--- SoA-related heading hits (full doc) ---")
for item, level in doc.iterate_items():
    if not isinstance(item, TextItem):
        continue
    text = (item.text or "").strip()
    lower = text.lower()
    if any(kw in lower for kw in SOA_HEADING_KEYWORDS):
        print(f"p{item_page(item)}  L{level}  {item_label(item)}  {text[:200]}")

000  L1  p1  text                  The information contained in this clinical study protocol is Copyright © 2006 Eli Lilly and Company.
001  L1  p1  section_header        Xanomeline (LY246708)
002  L1  p1  section_header        Protocol H2Q-MC-LZZT(c)
003  L1  p1  section_header        Safety and Efficacy of the Xanomeline Transdermal Therapeutic System (TTS) in Patients with Mild to Moderate Alzheimer's…
004  L1  p3  section_header        Table of Contents (continued)
005  L1  p3  document_index        [TABLE]
006  L1  p4  section_header        Table of Contents (concluded)
007  L1  p4  document_index        [TABLE]
008  L1  p5  section_header        Safety and Efficacy of the Xanomeline Transdermal Therapeutic System (TTS) in Patients with Mild to Moderate Alzheimer's…
009  L1  p5  section_header        1. Introduction
010  L1  p5  text                  The M 1  muscarinic-cholinergic receptor is 1 of 5 characterized muscarinic-cholinergic receptor subtypes (Fisher and Ba…
011  L1  p

## 5. Table survey (SoA-relevant)

For each table: page, dimensions, preview, and a simple candidate-SoA flag (large grid).

In [6]:
MIN_SOA_ROWS = 5
MIN_SOA_COLS = 4


def table_dims(table: TableItem) -> tuple[int, int]:
    data = getattr(table, "data", None)
    grid = getattr(data, "grid", None) if data else None
    if grid:
        rows = len(grid)
        cols = max((len(row) for row in grid), default=0)
        return rows, cols
    try:
        df = table.export_to_dataframe(doc=doc)
        return df.shape
    except Exception:
        return 0, 0


def table_caption(table: TableItem) -> str:
    cap = getattr(table, "caption", None)
    if cap is None:
        return ""
    if isinstance(cap, str):
        return cap
    return getattr(cap, "text", str(cap)) or ""


table_rows: list[dict] = []

for idx, table in enumerate(doc.tables, start=1):
    rows, cols = table_dims(table)
    page = item_page(table)
    caption = table_caption(table)
    candidate = rows >= MIN_SOA_ROWS and cols >= MIN_SOA_COLS

    table_rows.append(
        {
            "table": idx,
            "page": page,
            "rows": rows,
            "cols": cols,
            "candidate_soa": candidate,
            "caption": caption[:80],
        }
    )

    flag = " *** CANDIDATE SoA ***" if candidate else ""
    print(f"\nTable {idx}  p{page}  {rows}x{cols}{flag}")
    if caption:
        print(f"  caption: {caption[:200]}")

    try:
        df = table.export_to_dataframe(doc=doc)
        display(df.head(6))
    except Exception as exc:
        print(f"  (could not export dataframe: {exc})")

table_summary_df = pd.DataFrame(table_rows)
if not table_summary_df.empty:
    table_summary_sorted = table_summary_df.sort_values(
        ["candidate_soa", "rows", "cols"], ascending=False
    )
    print("\n--- Table summary ---")
    display(table_summary_sorted)

    summary_path = OUTPUT_DIR / f"{PROTOCOL.stem}-table-summary.csv"
    table_summary_sorted.to_csv(summary_path, index=False)
    print(f"\nSaved: {summary_path}")

    try:
        table_summary_sorted.to_clipboard(index=False)
        print("Copied to clipboard (tab-separated — paste into Excel/Sheets/Notes).")
    except Exception as exc:
        print(f"Clipboard copy failed ({exc}). Copy from the CSV file or text below:")
        print(table_summary_sorted.to_csv(index=False))


Table 1  p3  33x2


,Section,Page
0,3.9.3. Safety,.................................................
1,3.9.3.1. Safety Measures.........................,
2,3.9.3.2. Clinical Adverse Events.................,
3,3.9.3.2.1. Adverse Event Reporting Requirement...,
4,3.9.3.2.2. Serious Adverse Events ...............,
5,3.9.3.3. Clinical Laboratory Tests...............,



Table 2  p4  17x2


,Section,Page
0,"5. Informed Consent, Ethical Review, and Regul...",
1,5.1. Informed Consent............................,
2,5.2. Ethical Review,.................................................
3,5.3. Regulatory Considerations...................,
4,6. References....................................,
5,List of Attachments,



Table 3  p20  4x2


...


Table 21  p94  14x3


,Feature,Present,Absent
0,1. Abrupt onset,2,0
1,2. Stepwise deterioration,1,0
2,3. Fluctuating course,2,0
3,4. Nocturnal confusion,1,0
4,5. Relative preservation of personality,1,0
5,6. Depression,1,0



--- Table summary ---


,table,page,rows,cols,candidate_soa,caption
6,7,53,30,9,True,
7,8,54,30,9,True,
12,13,71,15,4,True,
4,5,37,7,4,True,
0,1,3,33,2,False,
1,2,4,17,2,False,
13,14,72,14,3,False,
20,21,94,14,3,False,
9,10,68,11,2,False,
10,11,69,10,2,False,



Saved: /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs/protocol1-table-summary.csv
Copied to clipboard (tab-separated — paste into Excel/Sheets/Notes).


## 6. Footnote detection (rule-based scoring)

Footnotes are load-bearing for SoA extraction, and Docling's `footnote` label undercounts them badly
(4 hits in this document, only 1 of which is in the SoA legend). This cell ignores the label entirely
and scores blocks structurally — see [`footnote_detect.md`](../footnote_detect.md) for the full design.

| signal | pts | check |
|---|---|---|
| R1 | +1 | leading token is a single-class run (`*`, `**`, `a`, `CT`, `2`) or X+letter (`Xa`) |
| | | …the token may arrive split in two (`X a`, `b X`) — a superscript is its own text run |
| R2 | +1 | token followed by a separator (`- – — = :`) or run on into prose |
| R3 | +1 | part of a run of 2+ consecutive marker-lines |
| R4 | +1 | block sits immediately below a table (this page, or one that ran off the previous page) |
| R5 | +2 | a block marker recurs inside that table's cells |

Score ≥ 6, or R5 plus 2 other signals → **accept**. 3–5 → **review**. < 3 → **discard**.
Every candidate is printed with its score, signals, attached table, and per-line markers.

In [7]:
# --- Footnote detection: structural scoring, no label trust, no template strings ---
import re

# Scoring weights — tune after reviewing all 5 protocols
W_R1_TOKEN_SHAPE = 1   # leading token is a single-class run, or X+letter
W_R2_SEPARATOR   = 1   # token followed by a separator, or run on into prose
W_R3_CLUSTER     = 1   # part of a run of 2+ marker-lines
W_R4_AFTER_TABLE = 1   # block sits immediately below a table
W_R5_IN_TABLE    = 2   # a block marker recurs inside that table

ACCEPT_SCORE        = 6   # >= this -> auto-accept
REVIEW_SCORE        = 3   # 3..5 -> flag for human review; < 3 -> discard
ACCEPT_R5_MIN_OTHER = 2   # R5 plus this many other signals also auto-accepts

MAX_TABLE_GAP_PT    = 80    # table bottom -> block top
LINE_Y_TOL_PT       = 3     # y tolerance when merging items into one visual line
MAX_INTERRUPT_LINES = 1     # non-candidate lines tolerated inside a block
MIN_CLUSTER_LINES   = 2     # R3 threshold
PAGE_TOP_FRAC       = 0.75  # "top of page" for page-break continuation
PAGE_BOTTOM_FRAC    = 0.25  # "table ends at page bottom"

SEPARATORS   = "-–—=:"
SYMBOLS      = "*†‡§¶#•"
SKIP_LABELS  = {"page_footer", "page_header", "picture"}
BREAK_LABELS = {"section_header", "title", "table"}

MAX_MARKER_FRAGS = 2        # a marker may arrive split in two ("X a", "b X")
MAX_FRAG_LEN     = 2        # ...each fragment that short — longer means it is a word

# Head is one token, or two short ones: the superscript is its own text run and may sort
# either side of its base glyph.
_SEP_SPLIT = re.compile(rf"^(\S+(?:\s+\S+)?)\s*([{re.escape(SEPARATORS)}])\s+(\S.*)$")
_SYM_RUNON = re.compile(rf"^([{re.escape(SYMBOLS)}]+)\s*(\S.*)$")
_GLUED_SYM = re.compile(rf"([{re.escape(SYMBOLS)}]+)$")   # "tests**" -> "**"


def _token_shape_ok(tok: str) -> bool:
    """R1: leading token is a single character class, or X+letter."""
    if tok and tok[0] in SYMBOLS and all(c == tok[0] for c in tok):
        return True                      # *, **, *****, †, ‡
    if tok.isalpha():
        if tok.isupper() or tok.islower():
            return True                  # a, aa, A, CT, RT (uniform case, any length)
        return len(tok) == 2 and tok[0].isupper() and tok[1].islower()   # Xa, Xb
    if tok.isdigit():
        return len(tok) <= 2             # 1, 2, 12 — "1." never gets here (no separator)
    return False                         # (06), 3a, mixed-case words


def norm_marker(head: str) -> str | None:
    """Canonical marker for a line/cell head, or None.

    A superscript is a separate text run in the PDF, so "X^a" reaches us as "X a" — and when the
    raised glyph sorts first, as "b X". Both are the marker Xa/Xb. Join the fragments (either
    order) and keep whichever passes the shape test.
    """
    frags = head.split()
    if len(frags) == 1:
        return frags[0] if _token_shape_ok(frags[0]) else None
    if len(frags) > MAX_MARKER_FRAGS or any(len(f) > MAX_FRAG_LEN for f in frags):
        return None                      # two real words, not a split marker
    for cand in ("".join(frags), "".join(reversed(frags))):
        if _token_shape_ok(cand):
            return cand
    return None


def marker_token(line: str):
    """(token, has_separator) if the line opens like a footnote marker, else None."""
    line = line.strip()
    m = _SEP_SPLIT.match(line)
    if m:
        tok = norm_marker(m.group(1))
        if tok:
            return tok, True
    m = _SYM_RUNON.match(line)           # run-on: *Days -15 through -9 are allotted...
    if m and _token_shape_ok(m.group(1)):
        return m.group(1), False
    return None


def page_lines(doc):
    """{page: [line dicts, top->bottom]} — items sharing a y-band merge into one visual line."""
    raw: dict[int, list[dict]] = {}
    for item, _lvl in doc.iterate_items():
        if not isinstance(item, TextItem) or item_label(item) in SKIP_LABELS:
            continue
        prov = (getattr(item, "prov", None) or [None])[0]
        if prov is None:
            continue
        bbox = prov.bbox
        for part in (item.text or "").split("\n"):
            if part.strip():
                raw.setdefault(prov.page_no, []).append(
                    {"text": part.strip(), "label": item_label(item),
                     "t": bbox.t, "b": bbox.b, "l": bbox.l, "r": bbox.r}
                )

    lines: dict[int, list[dict]] = {}
    for page, items in raw.items():
        items.sort(key=lambda d: (-d["t"], d["l"]))
        merged: list[dict] = []
        for it in items:
            prev = merged[-1] if merged else None
            if prev and abs(prev["t"] - it["t"]) <= LINE_Y_TOL_PT:
                prev["text"] = f"{prev['text']} {it['text']}"
                prev["r"] = max(prev["r"], it["r"])
                prev["b"] = min(prev["b"], it["b"])
            else:
                merged.append(dict(it))
        lines[page] = merged
    return lines


def page_tables(doc):
    """{page: [table dicts]} with bbox (BOTTOMLEFT) and flat cell texts."""
    out: dict[int, list[dict]] = {}
    for idx, table in enumerate(doc.tables, start=1):
        prov = (getattr(table, "prov", None) or [None])[0]
        if prov is None:
            continue
        cells = getattr(getattr(table, "data", None), "table_cells", []) or []
        out.setdefault(prov.page_no, []).append(
            {"id": f"table-{idx}", "page": prov.page_no,
             "t": prov.bbox.t, "b": prov.bbox.b,
             "cell_texts": [(c.text or "").strip() for c in cells]}
        )
    return out


def _close(entries: list[dict]) -> dict:
    while entries and not entries[-1]["marker"]:
        entries.pop()                    # trim trailing interrupt line
    return {"page": entries[0]["page"], "lines": entries,
            "top": max(e["t"] for e in entries),
            "bottom": min(e["b"] for e in entries),
            "markers": [e["marker"] for e in entries if e["marker"]]}


def build_blocks(lines_by_page) -> list[dict]:
    """Contiguous runs of marker-lines, tolerating MAX_INTERRUPT_LINES non-candidates."""
    blocks: list[dict] = []
    for page, lines in sorted(lines_by_page.items()):
        cur: list[dict] = []
        gap = 0

        def flush():
            nonlocal cur, gap
            if any(e["marker"] for e in cur):
                blocks.append(_close(cur))
            cur, gap = [], 0

        for ln in lines:
            if ln["label"] in BREAK_LABELS:      # heading breaks the sequence
                flush()
                continue
            hit = marker_token(ln["text"])
            entry = {**ln, "page": page,
                     "marker": hit[0] if hit else None,
                     "has_sep": hit[1] if hit else False}
            if hit:
                cur.append(entry)
                gap = 0
            elif cur and gap < MAX_INTERRUPT_LINES:
                cur.append(entry)
                gap += 1
            else:
                flush()
        flush()
    return blocks


def attach_table(block, tables_by_page, page_heights):
    """R4: nearest table above on this page, else one that ran to the previous page's bottom."""
    same = [t for t in tables_by_page.get(block["page"], [])
            if 0 <= t["b"] - block["top"] <= MAX_TABLE_GAP_PT]
    if same:
        return min(same, key=lambda t: t["b"] - block["top"]), "same-page"

    prev = tables_by_page.get(block["page"] - 1, [])
    height = page_heights.get(block["page"]) or 792.0
    if prev and block["top"] >= height * PAGE_TOP_FRAC:
        last = min(prev, key=lambda t: t["b"])
        prev_height = page_heights.get(block["page"] - 1) or 792.0
        if last["b"] <= prev_height * PAGE_BOTTOM_FRAC:
            return last, "page-break"
    return None, None


# SoA cell values are X-family: the value is X (sometimes count-prefixed, 2X / 3X) and the marker is a
# superscript riding on it. Docling sometimes flattens that superscript into the text ("Xa", "3Xb")
# and more often drops it, so match on the stripped value rather than expecting a bare marker token.
_CELL_VALUE = re.compile(r"^(?P<count>\d{1,2})?\s*(?P<base>[A-Za-z]{1,2})(?P<suffix>[a-z])?$")


def cell_tokens(cell_texts) -> set[str]:
    """Every form a marker can take inside a cell: the cell, its words, adjacent short-word pairs
    joined both ways ("X a" -> Xa), and a symbol run glued to a word ("tests**" -> **)."""
    tokens: set[str] = set()
    for txt in cell_texts:
        txt = txt.strip()
        if not txt:
            continue
        words = txt.split()
        tokens.add(txt)
        tokens.update(words)
        for a, b in zip(words, words[1:]):
            if len(a) <= MAX_FRAG_LEN and len(b) <= MAX_FRAG_LEN:
                tokens.update((a + b, b + a))
        for w in words:
            m = _GLUED_SYM.search(w)
            if m:
                tokens.add(m.group(1))
    return tokens


def cell_marker_hits(cell_texts, markers) -> list[str]:
    """R5: markers that recur inside the table."""
    tokens = cell_tokens(cell_texts)
    hits = set()
    for m in markers:
        if m in tokens:
            hits.add(m)
            continue
        for tok in tokens:
            mm = _CELL_VALUE.match(tok)
            if not mm or not mm.group("base").isupper():
                continue                 # guard: keeps marker "a" off words like "Data"
            if mm.group("suffix") == m or mm.group("base") == m:
                hits.add(m)
                break
    return sorted(hits)


def score_block(block, table):
    marker_lines = [e for e in block["lines"] if e["marker"]]
    sig = {
        "R1": W_R1_TOKEN_SHAPE if marker_lines else 0,
        "R2": W_R2_SEPARATOR if any(e["has_sep"] or e["marker"][0] in SYMBOLS
                                    for e in marker_lines) else 0,
        "R3": W_R3_CLUSTER if len(marker_lines) >= MIN_CLUSTER_LINES else 0,
        "R4": W_R4_AFTER_TABLE if table else 0,
    }
    linked = cell_marker_hits(table["cell_texts"], block["markers"]) if table else []
    sig["R5"] = W_R5_IN_TABLE if linked else 0

    score = sum(sig.values())
    others = sum(1 for k in ("R1", "R2", "R3", "R4") if sig[k])
    if score >= ACCEPT_SCORE or (sig["R5"] and others >= ACCEPT_R5_MIN_OTHER):
        verdict = "accept"
    elif score >= REVIEW_SCORE:
        verdict = "review"
    else:
        verdict = "discard"
    return score, sig, verdict, linked


def detect_footnotes(doc) -> list[dict]:
    """Scored footnote blocks for one document — the entry point cell 9 reuses per protocol."""
    tables_by_page = page_tables(doc)
    page_heights = {no: getattr(getattr(pg, "size", None), "height", 792.0)
                    for no, pg in (getattr(doc, "pages", {}) or {}).items()}

    out = []
    for blk in build_blocks(page_lines(doc)):
        table, how = attach_table(blk, tables_by_page, page_heights)
        score, sig, verdict, linked = score_block(blk, table)
        out.append({
            "block": blk, "table": table, "attach": how,
            "score": score, "signals": sig, "verdict": verdict,
            "linked": linked,
            "unlinked": sorted({m for m in blk["markers"] if m not in linked}),
        })
    return out


results = detect_footnotes(doc)

for verdict in ("accept", "review", "discard"):
    group = [r for r in results if r["verdict"] == verdict]
    print(f"\n{'=' * 90}\n{verdict.upper()}  ({len(group)} blocks)\n{'=' * 90}")
    for r in group:
        blk = r["block"]
        hits = " ".join(f"{k}={v}" for k, v in r["signals"].items())
        tbl = f"{r['table']['id']} ({r['attach']})" if r["table"] else "none"
        print(f"\np{blk['page']}  y {blk['top']:.0f}->{blk['bottom']:.0f}  "
              f"score={r['score']}  {hits}")
        print(f"  table: {tbl}   markers: {blk['markers']}")
        print(f"  linked: {r['linked']}   unlinked: {r['unlinked']}")
        for e in blk["lines"]:
            tag = f"[{e['marker']}]" if e["marker"] else "[ - ]"
            print(f"    {tag:8s} {e['label']:14s} {e['text'][:150]}")

summary = pd.DataFrame([{
    "document": PROTOCOL.name,
    "accepted": sum(r["verdict"] == "accept" for r in results),
    "flagged": sum(r["verdict"] == "review" for r in results),
    "discarded": sum(r["verdict"] == "discard" for r in results),
    "markers": sum(len(r["block"]["markers"]) for r in results if r["verdict"] != "discard"),
    "markers_linked": sum(len(r["linked"]) for r in results if r["verdict"] != "discard"),
    "markers_unlinked": sum(len(r["unlinked"]) for r in results if r["verdict"] != "discard"),
    "docling_footnote_labels": sum(
        1 for it, _ in doc.iterate_items()
        if item_label(it) == DocItemLabel.FOOTNOTE.value
    ),
}])
display(summary)


ACCEPT  (2 blocks)

p53  y 152->79  score=6  R1=1 R2=1 R3=1 R4=1 R5=2
  table: table-7 (same-page)   markers: ['X', 'Xa', 'Xb', 'P']
  linked: ['P', 'X', 'Xa', 'Xb']   unlinked: []
    [X]      text           X = Performed at this visit.
    [Xa]     text           Xa  = Performed at this visit if patient is an insulin-dependent diabetic.
    [Xb]     text           Xb   = Performed at this visit and via telephone interview 2 weeks following this visit.
    [P]      footnote       P = Practice only - It is recommended that a sampling of the CIBIC+, ADAS-Cog, DAD, and NPI-X be administered at Visit 1.  Data from this sampling wou

p54  y 192->171  score=6  R1=1 R2=1 R3=1 R4=1 R5=2
  table: table-8 (same-page)   markers: ['X', 'Xb']
  linked: ['X', 'Xb']   unlinked: []
    [X]      text           X = Performed at this visit.
    [Xb]     text           Xb   = Performed at this visit and via telephone interview 2 weeks following this visit.

REVIEW  (2 blocks)

p58  y 435->336  score=3  

,document,accepted,flagged,discarded,markers,markers_linked,markers_unlinked,docling_footnote_labels
0,protocol1.pdf,2,2,10,14,6,8,4


## 7. SoA table identification

Docling has parsed the document and cell 6 has found the footnote blocks. This step picks the **target
SoA table(s)** out of every table on the page — scored, not matched on headings. Full architecture in
[`footnote_detect.md`](../footnote_detect.md) §6.

| rule | pts | check |
|---|---|---|
| 1 | +1 | **X-value share** — body cells matching `^\d*X$` (`X`, `2X`, `3X`…) over all non-empty body cells > 90% |
| 2 | +0.5 | **Footnote block present** — a block from cell 6 attached to this table, or a short legend header line under it |
| 3 | +0.5 | **Density** — `empty/non_empty` < 0.80 **and** `non_empty/empty` > 2 |
| 4 | gate | **Timepoint axis** — a header row or the label column carries ordered timepoint-like labels; failing this disqualifies the table outright |

Max 2.0, and `SOA_SCORE_THRESHOLD = 2.0` — all three scored rules must fire on top of the gate. Every
eligible table at or above it is returned — a protocol may hold more than one SoA (main + sub-study,
PK sub-schedule, LTE). Rejected tables are printed with the reason.

In [8]:
# --- SoA table identification: scored over every Docling table, gated on a timepoint axis ---
# Runs after footnote detection: rule 2 consumes `results` / `page_lines` from the cell above.

W_X_VALUE  = 1.0    # rule 1: body cells are overwhelmingly X-type markers
W_FOOTNOTE = 0.5    # rule 2: a footnote block hangs off this table
W_DENSITY  = 0.5    # rule 3: table is dense rather than mostly empty

X_VALUE_MIN_PCT         = 0.90   # rule 1 threshold: x_values / non_empty
DENSITY_MAX_EMPTY_RATIO = 0.80   # rule 3: empty / non_empty must be below this
DENSITY_MIN_FILL_RATIO  = 2.0    # rule 3: non_empty / empty must exceed this
MIN_TIMEPOINT_LABELS    = 3      # rule 4 gate: ordered timepoint-like labels needed on an axis
FOOTNOTE_HEADER_CHARS   = 60     # a short line under a table that reads as a legend header
SOA_SCORE_THRESHOLD     = 2.0    # report every eligible table at or above this (max 2.0)

# "X", "2X", "3X", "6X" — the leading digits are a repetition count, the cell is still an X marker.
_X_VALUE = re.compile(r"^\d*X$")
# Generic timepoint vocabulary — study-design words, not sponsor or template strings.
_TIMEPOINT_WORD = re.compile(
    r"\b(visit|day|week|month|year|cycle|hour|screen\w*|baseline|random\w*|"
    r"treatment|follow[\s-]?up|end of|eot|eos|unscheduled|termination|period|epoch)\b",
    re.IGNORECASE,
)
_INT = re.compile(r"-?\d+")


def table_grid(table):
    return getattr(getattr(table, "data", None), "grid", None) or []


def split_axes(grid):
    """(header row indices, row-label column indices).

    Docling's `column_header` flag is unreliable on stacked headers — on protocol1's SoA it flags the
    VISIT row but not the WEEK row under it, which leaks 8 timepoint labels into the body and drags
    the x-value ratio down by ~9 points. So: take the flagged rows, then keep walking down while a row
    still reads as header (timepoint-like labels, no body value in it).
    """
    header_rows = {r for r, row in enumerate(grid)
                   if any(getattr(c, "column_header", False) for c in row)}
    label_cols = {i for row in grid for i, c in enumerate(row)
                  if getattr(c, "row_header", False)}
    if not label_cols and grid:
        label_cols = {0}          # unflagged: first column carries the activity labels

    for r, row in enumerate(grid):
        if r in header_rows:
            continue
        cells = [(getattr(c, "text", "") or "").strip()
                 for i, c in enumerate(row) if i not in label_cols]
        cells = [c for c in cells if c]
        if cells and not any(_X_VALUE.match(c) for c in cells) \
                and all(_TIMEPOINT_WORD.search(c) or _INT.search(c) for c in cells):
            header_rows.add(r)
            continue
        break                     # first real body row ends the header stack
    return header_rows, label_cols


def body_texts(grid) -> list[str]:
    header_rows, label_cols = split_axes(grid)
    out = []
    for r, row in enumerate(grid):
        if r in header_rows:
            continue
        for i, cell in enumerate(row):
            if i in label_cols or getattr(cell, "row_section", False):
                continue
            out.append((getattr(cell, "text", "") or "").strip())
    return out


def timepoint_axis(grid) -> tuple[bool, str]:
    """Rule 4 gate: some header row or the label column holds ordered timepoint-like labels."""
    header_rows, label_cols = split_axes(grid)
    axes: list[tuple[str, list[str]]] = [
        (f"header row {r}", [(getattr(c, "text", "") or "").strip() for c in grid[r]])
        for r in sorted(header_rows)
    ]
    for i in sorted(label_cols):
        axes.append((f"column {i}",
                     [(getattr(row[i], "text", "") or "").strip()
                      for row in grid if i < len(row)]))

    for name, cells in axes:
        cells = [c for c in cells if c]
        words = sum(1 for c in cells if _TIMEPOINT_WORD.search(c))
        if words >= MIN_TIMEPOINT_LABELS:
            return True, f"{name}: {words} timepoint labels"
        nums = [int(m.group()) for c in cells for m in [_INT.search(c)] if m]
        if len(nums) >= MIN_TIMEPOINT_LABELS and len(set(nums)) >= MIN_TIMEPOINT_LABELS:
            if nums == sorted(nums) or nums == sorted(nums, reverse=True):
                return True, f"{name}: ordered numeric sequence {nums[:6]}…"
    return False, "no ordered timepoint axis"


def footnote_evidence(table_id, page, bottom, blocks, lines_by_page):
    """Rule 2: a detected footnote block attached to this table, or a legend header under it."""
    attached = [r for r in blocks
                if r["table"] and r["table"]["id"] == table_id and r["verdict"] != "discard"]
    if attached:
        markers = sorted({m for r in attached for m in r["block"]["markers"]})
        return True, f"{len(attached)} block(s), markers {markers}", attached
    for ln in lines_by_page.get(page, []):
        if 0 <= bottom - ln["t"] <= MAX_TABLE_GAP_PT \
                and len(ln["text"]) <= FOOTNOTE_HEADER_CHARS \
                and ln["text"].rstrip().endswith(":"):
            return True, f"legend header {ln['text']!r}", []
    return False, "none", []


lines_by_page = page_lines(doc)
soa_rows: list[dict] = []
soa_detail: dict[str, dict] = {}

for idx, table in enumerate(doc.tables, start=1):
    table_id = f"table-{idx}"
    grid = table_grid(table)
    prov = (getattr(table, "prov", None) or [None])[0]
    page = prov.page_no if prov else None

    texts = body_texts(grid)
    non_empty = [t for t in texts if t]
    empty = len(texts) - len(non_empty)
    x_values = [t for t in non_empty if _X_VALUE.match(t)]

    x_pct = len(x_values) / len(non_empty) if non_empty else 0.0
    empty_ratio = empty / len(non_empty) if non_empty else float("inf")
    fill_ratio = len(non_empty) / empty if empty else float("inf")

    eligible, gate_why = timepoint_axis(grid)
    has_fn, fn_why, fn_blocks = footnote_evidence(
        table_id, page, prov.bbox.b if prov else 0.0, results, lines_by_page)

    sig = {
        "R1_x_values": W_X_VALUE if x_pct > X_VALUE_MIN_PCT else 0.0,
        "R2_footnotes": W_FOOTNOTE if has_fn else 0.0,
        "R3_density": W_DENSITY if (empty_ratio < DENSITY_MAX_EMPTY_RATIO
                                    and fill_ratio > DENSITY_MIN_FILL_RATIO) else 0.0,
    }
    score = sum(sig.values()) if eligible else 0.0

    soa_rows.append({
        "table": table_id, "page": page,
        "rows": len(grid), "cols": max((len(r) for r in grid), default=0),
        "body_cells": len(texts), "non_empty": len(non_empty),
        "x_pct": round(x_pct, 3), "empty_ratio": round(empty_ratio, 2),
        "eligible": eligible, "score": score,
        "R1": sig["R1_x_values"], "R2": sig["R2_footnotes"], "R3": sig["R3_density"],
        "soa": eligible and score >= SOA_SCORE_THRESHOLD,
    })
    soa_detail[table_id] = {"gate_why": gate_why, "fn_why": fn_why, "fn_blocks": fn_blocks,
                            "sample": [t for t in non_empty[:12]]}

soa_df = pd.DataFrame(soa_rows).sort_values(
    ["soa", "score", "x_pct"], ascending=False).reset_index(drop=True)

print(f"Tables parsed by Docling: {len(doc.tables)}   "
      f"footnote blocks accepted/flagged: "
      f"{sum(1 for r in results if r['verdict'] != 'discard')}")
print(f"Eligible (rule 4 gate passed): {sum(r['eligible'] for r in soa_rows)}   "
      f"SoA candidates (score >= {SOA_SCORE_THRESHOLD}): {sum(r['soa'] for r in soa_rows)}\n")
display(soa_df)

print(f"\n{'=' * 90}\nSoA CANDIDATES\n{'=' * 90}")
for row in soa_df[soa_df["soa"]].to_dict("records"):
    d = soa_detail[row["table"]]
    print(f"\n{row['table']}  p{row['page']}  {row['rows']}x{row['cols']}  score={row['score']}")
    print(f"  R1 x-values {row['x_pct']:.0%} of {row['non_empty']} non-empty  -> {row['R1']}")
    print(f"  R2 footnotes {d['fn_why']}  -> {row['R2']}")
    print(f"  R3 empty/non_empty {row['empty_ratio']}  -> {row['R3']}")
    print(f"  gate: {d['gate_why']}")
    print(f"  sample body values: {d['sample']}")
    for r in d["fn_blocks"]:
        blk = r["block"]
        print(f"  footnote block p{blk['page']} ({r['verdict']}, score {r['score']}, "
              f"{r['attach']}): markers {blk['markers']}")
        for e in blk["lines"]:
            if e["marker"]:
                print(f"      [{e['marker']}] {e['text'][:120]}")

rejected = soa_df[~soa_df["soa"]]
print(f"\n{'=' * 90}\nNOT SELECTED ({len(rejected)})\n{'=' * 90}")
for row in rejected.to_dict("records"):
    d = soa_detail[row["table"]]
    why = d["gate_why"] if not row["eligible"] else (
        f"score {row['score']} < {SOA_SCORE_THRESHOLD} "
        f"(x {row['x_pct']:.0%}, footnotes {d['fn_why']}, empty/non_empty {row['empty_ratio']})")
    print(f"{row['table']:9s} p{row['page']:<4} {row['rows']:>3}x{row['cols']:<3} "
          f"{'gated out' if not row['eligible'] else 'low score':10s}  {why}")

soa_path = OUTPUT_DIR / f"{PROTOCOL.stem}-soa-candidates.csv"
soa_df.to_csv(soa_path, index=False)
print(f"\nSaved: {soa_path}")

Tables parsed by Docling: 21   footnote blocks accepted/flagged: 4
Eligible (rule 4 gate passed): 5   SoA candidates (score >= 2.0): 0



,table,page,rows,cols,body_cells,non_empty,x_pct,empty_ratio,eligible,score,R1,R2,R3,soa
0,table-8,54,30,9,224,68,0.956,2.29,True,1.5,1.0,0.5,0.0,False
1,table-7,53,30,9,224,71,0.915,2.15,True,1.5,1.0,0.5,0.0,False
2,table-5,37,7,4,3,3,0.333,0.00,True,0.5,0.0,0.0,0.5,False
3,table-1,3,33,2,31,8,0.000,2.88,False,0.0,0.0,0.0,0.0,False
4,table-2,4,17,2,16,1,0.000,15.00,False,0.0,0.0,0.0,0.0,False
5,table-3,20,4,2,0,0,0.000,inf,False,0.0,0.0,0.0,0.0,False
6,table-4,25,8,2,7,7,0.000,0.00,False,0.0,0.0,0.0,0.5,False
7,table-6,38,7,2,0,0,0.000,inf,True,0.0,0.0,0.0,0.0,False
8,table-9,66,7,2,0,0,0.000,inf,False,0.0,0.0,0.0,0.0,False
9,table-10,68,11,2,10,10,0.000,0.00,False,0.0,0.0,0.0,0.5,False



SoA CANDIDATES

NOT SELECTED (21)
table-8   p54    30x9   low score   score 1.5 < 2.0 (x 96%, footnotes 1 block(s), markers ['X', 'Xb'], empty/non_empty 2.29)
table-7   p53    30x9   low score   score 1.5 < 2.0 (x 92%, footnotes 1 block(s), markers ['P', 'X', 'Xa', 'Xb'], empty/non_empty 2.15)
table-5   p37     7x4   low score   score 0.5 < 2.0 (x 33%, footnotes none, empty/non_empty 0.0)
table-1   p3     33x2   gated out   no ordered timepoint axis
table-2   p4     17x2   gated out   no ordered timepoint axis
table-3   p20     4x2   gated out   no ordered timepoint axis
table-4   p25     8x2   gated out   no ordered timepoint axis
table-6   p38     7x2   low score   score 0.0 < 2.0 (x 0%, footnotes none, empty/non_empty inf)
table-9   p66     7x2   gated out   no ordered timepoint axis
table-10  p68    11x2   gated out   no ordered timepoint axis
table-11  p69    10x2   gated out   no ordered timepoint axis
table-12  p70     7x2   gated out   no ordered timepoint axis
table-13  p71  

## 8. Export artifacts for manual review

In [9]:
stem = PROTOCOL.stem
md_path = OUTPUT_DIR / f"{stem}.md"
json_path = OUTPUT_DIR / f"{stem}-docling.json"
labels_path = OUTPUT_DIR / f"{stem}-labels.csv"

md_path.write_text(doc.export_to_markdown(), encoding="utf-8")

doc_dict = doc.export_to_dict()
json_path.write_text(json.dumps(doc_dict, indent=2, default=str), encoding="utf-8")

label_df.to_csv(labels_path, index=False)

print(f"Wrote {md_path}")
print(f"Wrote {json_path}")
print(f"Wrote {labels_path}")

Wrote /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs/protocol1.md
Wrote /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs/protocol1-docling.json
Wrote /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs/protocol1-labels.csv


## 9. Cross-protocol comparison (optional)

Runs Docling on all five assignment PDFs. **Slow** — expect several minutes total.

The `fn_*` columns come from `detect_footnotes()` (cell 6). `docling_footnote_labels` is the layout
classifier's own `footnote` count — reported only to show how far off it is, never as the score.


In [10]:
RUN_CROSS_PROTOCOL = True  # set True to run all five

if RUN_CROSS_PROTOCOL:
    pdfs = sorted(PROTOCOLS_DIR.glob("protocol*.pdf"))
    comparison: list[dict] = []

    for pdf in pdfs:
        t0 = time.perf_counter()
        res = converter.convert(str(pdf))
        elapsed = time.perf_counter() - t0
        d = res.document
        pages = len(getattr(d, "pages", {}) or {})

        labels = Counter()
        for item, _ in d.iterate_items():
            labels[item_label(item)] += 1

        # Docling's `footnote` label is not the detector — cell 6 ignores it on purpose.
        # Report both so the gap stays visible.
        fn = detect_footnotes(d)
        kept = [r for r in fn if r["verdict"] != "discard"]

        largest = (0, 0)
        for table in d.tables:
            r, c = table_dims(table)
            largest = max(largest, (r, c), key=lambda x: x[0] * x[1])

        comparison.append(
            {
                "pdf": pdf.name,
                "pages": pages,
                "tables": len(d.tables),
                "largest_table": f"{largest[0]}x{largest[1]}",
                "section_headers": labels.get(DocItemLabel.SECTION_HEADER.value, 0),
                "fn_blocks": len(kept),
                "fn_accepted": sum(r["verdict"] == "accept" for r in kept),
                "fn_markers": sum(len(r["block"]["markers"]) for r in kept),
                "fn_linked": sum(len(r["linked"]) for r in kept),
                "docling_footnote_labels": labels.get(DocItemLabel.FOOTNOTE.value, 0),
                "seconds": round(elapsed, 1),
            }
        )

    cmp_df = pd.DataFrame(comparison)
    display(cmp_df)
    cmp_path = OUTPUT_DIR / "cross-protocol-comparison.csv"
    cmp_df.to_csv(cmp_path, index=False)
    print(f"Wrote {cmp_path}")
else:
    print("Set RUN_CROSS_PROTOCOL = True to compare all five PDFs.")

2026-09-02 22:26:32,382 MatchingPostProcessor WARNING  Orphan pdf_cell 148 recovered to row=23 by nearest-row fallback (col=0, y=1044.4, dist=41.0)
2026-09-02 22:26:32,384 MatchingPostProcessor WARNING  Orphan pdf_cell 149 recovered to row=23 by nearest-row fallback (col=0, y=1044.4, dist=41.0)
2026-09-02 22:26:32,384 MatchingPostProcessor WARNING  Orphan pdf_cell 150 recovered to row=24 by nearest-row fallback (col=0, y=1070.4, dist=27.6)
2026-09-02 22:26:32,384 MatchingPostProcessor WARNING  Orphan pdf_cell 151 recovered to row=24 by nearest-row fallback (col=0, y=1070.4, dist=27.6)
2026-09-02 22:26:35,843 MatchingPostProcessor WARNING  Orphan pdf_cell 124 recovered to row=21 by nearest-row fallback (col=0, y=856.6, dist=25.9)
2026-09-02 22:26:52,998 MatchingPostProcessor WARNING  Orphan pdf_cell 46 recovered to col=2 by nearest-column fallback (row=6, x=176.5, dist=465.0)
2026-09-02 22:26:52,999 MatchingPostProcessor WARNING  Orphan pdf_cell 21 recovered to row=4 by nearest-row fall

,pdf,pages,tables,largest_table,section_headers,fn_blocks,fn_accepted,fn_markers,fn_linked,docling_footnote_labels,seconds
0,protocol1.pdf,97,22,30x9,196,4,2,14,6,4,32.6
1,protocol12.pdf,97,12,50x3,183,5,4,18,6,7,32.7
2,protocol15.pdf,61,12,45x12,161,3,2,8,5,5,36.5
3,protocol5.pdf,61,13,32x12,139,2,1,13,10,5,26.1
4,protocol9.pdf,57,12,20x12,83,2,0,5,0,0,20.0


Wrote /Users/rushi/Desktop/Rushi/Assessment/intake-soa/outputs/cross-protocol-comparison.csv
